# Wind Turbine Data Processing Pipeline

Ingests raw CSV sensor data from a wind farm, cleans it, computes daily summary statistics, detects anomalous turbine behaviour, and persists results to Delta tables.

**Incremental design:** CSVs are appended daily with the last 24 hours of data. The pipeline reads the full CSV, identifies new records not yet in the Delta table, and processes only the new data — merging results into existing tables.

**Data:** 3 CSV files, each containing hourly readings for a group of 5 turbines (15 total), recorded over March 2022.

## 1. Configuration

In [0]:
import os

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
workspace_dir = "/Workspace" + "/".join(notebook_path.split("/")[:-1])
DATA_DIR = f"file:{workspace_dir}/Data"

DATA_FILES = [
    f"{DATA_DIR}/data_group_1.csv",
    f"{DATA_DIR}/data_group_2.csv",
    f"{DATA_DIR}/data_group_3.csv",
]

CATALOG = "hive_metastore"
SCHEMA = "wind_turbine"

TABLE_CLEANED = f"{CATALOG}.{SCHEMA}.cleaned_readings"
TABLE_DAILY_SUMMARY = f"{CATALOG}.{SCHEMA}.daily_summary"
TABLE_ANOMALIES = f"{CATALOG}.{SCHEMA}.anomalies"

EXPECTED_COLUMNS = ["timestamp", "turbine_id", "wind_speed", "wind_direction", "power_output"]

VALID_RANGES = {
    "wind_speed": (0.0, 100.0),
    "wind_direction": (0, 360),
    "power_output": (0.0, 15.0),
}

ANOMALY_STD_THRESHOLD = 2.0
OUTLIER_STD_THRESHOLD = 3.0
SUMMARY_PERIOD_HOURS = 24

In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.cleaned_readings")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.daily_summary")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.anomalies")

DataFrame[]

## 2. Imports & Spark Session

In [0]:
from functools import reduce
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, TimestampType, IntegerType, DoubleType
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

## 3. Create Schema

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

## 4. Data Ingestion

Load CSV files, enforce schema, union into a single DataFrame, and extract only new records not yet present in the cleaned Delta table (incremental detection via anti-join).

In [0]:
CSV_SCHEMA = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("turbine_id", IntegerType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("wind_direction", DoubleType(), True),
    StructField("power_output", DoubleType(), True),
])


def load_csv(filepath: str) -> DataFrame:
    """Load a single CSV with schema enforcement."""
    df = (
        spark.read
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .schema(CSV_SCHEMA)
        .csv(filepath)
    )
    row_count = df.count()
    print(f"Loaded {row_count} records from {filepath.split('/')[-1]}")
    return df


def load_all(file_list: list) -> DataFrame:
    """Load and union all CSV files into one DataFrame."""
    frames = [load_csv(f) for f in file_list]
    combined = reduce(DataFrame.unionByName, frames)
    combined = combined.orderBy("turbine_id", "timestamp")

    total = combined.count()
    turbine_count = combined.select("turbine_id").distinct().count()
    print(f"Combined dataset: {total} records, {turbine_count} turbines")
    return combined


def extract_new_records(raw_df: DataFrame, table_name: str) -> DataFrame:
    """
    Compare the raw CSV data against the existing Delta table and return
    only the records that are not already stored. On first run (table does
    not exist yet) all records are treated as new.
    """
    if not spark.catalog.tableExists(table_name):
        print(f"Table '{table_name}' does not exist yet — all records are new")
        return raw_df

    existing = spark.table(table_name).select("turbine_id", "timestamp")

    new_records = raw_df.join(
        existing,
        on=["turbine_id", "timestamp"],
        how="left_anti",
    )

    new_count = new_records.count()
    print(f"Identified {new_count} new records (out of {raw_df.count()} in CSVs)")
    return new_records

## 5. Data Cleaning

Four-pass cleaning using Spark: duplicate removal, range validation, missing value imputation (forward/backward fill via window functions), and outlier capping.

In [0]:
def remove_duplicates(df: DataFrame) -> DataFrame:
    """Drop duplicate readings for the same turbine and timestamp."""
    before = df.count()
    df = df.dropDuplicates(["turbine_id", "timestamp"])
    dropped = before - df.count()
    if dropped:
        print(f"Removed {dropped} duplicate records")
    return df


def replace_out_of_range(df: DataFrame) -> DataFrame:
    """Replace values outside physically plausible ranges with null."""
    for col, (low, high) in VALID_RANGES.items():
        df = df.withColumn(
            col,
            F.when(F.col(col).between(low, high), F.col(col)).otherwise(F.lit(None))
        )
    return df


def impute_missing(df: DataFrame) -> DataFrame:
    """
    Fill missing numeric values using forward fill then backward fill
    within each turbine, ordered by timestamp.
    """
    w_forward = Window.partitionBy("turbine_id").orderBy("timestamp").rowsBetween(Window.unboundedPreceding, 0)
    w_backward = Window.partitionBy("turbine_id").orderBy("timestamp").rowsBetween(0, Window.unboundedFollowing)

    numeric_cols = ["wind_speed", "wind_direction", "power_output"]

    for col in numeric_cols:
        df = df.withColumn(col, F.last(col, ignorenulls=True).over(w_forward))
        df = df.withColumn(col, F.first(col, ignorenulls=True).over(w_backward))

    return df


def remove_statistical_outliers(df: DataFrame, n_std: float = OUTLIER_STD_THRESHOLD) -> DataFrame:
    """
    Cap power_output values beyond n_std standard deviations of the
    per-turbine mean using winsorisation.
    """
    stats = df.groupBy("turbine_id").agg(
        F.mean("power_output").alias("_mean"),
        F.stddev("power_output").alias("_std"),
    )

    df = df.join(stats, on="turbine_id", how="left")

    df = df.withColumn(
        "power_output",
        F.greatest(
            F.col("_mean") - n_std * F.col("_std"),
            F.least(F.col("power_output"), F.col("_mean") + n_std * F.col("_std"))
        )
    )

    df = df.drop("_mean", "_std")
    return df


def clean(df: DataFrame) -> DataFrame:
    """Run the full cleaning pipeline."""
    df = remove_duplicates(df)
    df = replace_out_of_range(df)
    df = impute_missing(df)
    df = remove_statistical_outliers(df)
    return df

## 6. Summary Statistics

Compute min, max, and mean power output per turbine per calendar day, plus an overall summary across the full period.

In [0]:
def compute_daily_summary(df: DataFrame) -> DataFrame:
    """
    Compute min, max, and mean power output for each turbine
    over 24-hour windows aligned to calendar dates.
    """
    df_with_date = df.withColumn("date", F.to_date("timestamp"))

    summary = (
        df_with_date.groupBy("turbine_id", "date")
        .agg(
            F.min("power_output").alias("min_power"),
            F.max("power_output").alias("max_power"),
            F.round(F.mean("power_output"), 4).alias("mean_power"),
            F.count("power_output").alias("record_count"),
        )
        .orderBy("turbine_id", "date")
    )

    incomplete = summary.filter(F.col("record_count") < SUMMARY_PERIOD_HOURS).count()
    if incomplete:
        print(
            f"{incomplete} turbine-day combinations have fewer than "
            f"{SUMMARY_PERIOD_HOURS} readings (possible sensor gaps)"
        )

    print(f"Generated {summary.count()} daily summaries")
    return summary


def compute_overall_summary(df: DataFrame) -> DataFrame:
    """Aggregate statistics across the entire reporting period per turbine."""
    return (
        df.groupBy("turbine_id")
        .agg(
            F.min("power_output").alias("min_power"),
            F.max("power_output").alias("max_power"),
            F.round(F.mean("power_output"), 4).alias("mean_power"),
            F.round(F.stddev("power_output"), 4).alias("std_power"),
            F.count("power_output").alias("total_records"),
        )
        .orderBy("turbine_id")
    )

## 7. Anomaly Detection

Flag readings where a turbine's power output deviates by more than 2 standard deviations from its own daily mean.

In [0]:
def detect_anomalies(df: DataFrame, threshold: float = ANOMALY_STD_THRESHOLD) -> DataFrame:
    """
    Flag individual readings where a turbine's power output deviates
    by more than `threshold` standard deviations from its own daily mean.
    """
    df_with_date = df.withColumn("date", F.to_date("timestamp"))

    daily_stats = (
        df_with_date.groupBy("turbine_id", "date")
        .agg(
            F.mean("power_output").alias("daily_mean"),
            F.stddev("power_output").alias("daily_std"),
        )
    )

    merged = df_with_date.join(daily_stats, on=["turbine_id", "date"], how="left")

    merged = merged.withColumn(
        "z_score",
        F.when(
            F.col("daily_std") > 0,
            F.round((F.col("power_output") - F.col("daily_mean")) / F.col("daily_std"), 4)
        ).otherwise(0.0)
    )

    anomalies = merged.filter(F.abs(F.col("z_score")) > threshold)

    anomalies = anomalies.withColumn(
        "deviation_mw",
        F.round(F.col("power_output") - F.col("daily_mean"), 4)
    )

    result = anomalies.select(
        "timestamp", "turbine_id", "power_output", "wind_speed",
        F.round("daily_mean", 4).alias("daily_mean"),
        F.round("daily_std", 4).alias("daily_std"),
        "z_score", "deviation_mw",
    ).orderBy("turbine_id", "timestamp")

    count = result.count()
    turbines = result.select("turbine_id").distinct().count()
    print(f"Detected {count} anomalous readings across {turbines} turbines")
    return result


def summarise_anomalies(anomalies: DataFrame) -> DataFrame:
    """Aggregate anomaly counts and severity per turbine."""
    return (
        anomalies.groupBy("turbine_id")
        .agg(
            F.count("z_score").alias("anomaly_count"),
            F.round(F.mean("deviation_mw"), 4).alias("avg_deviation_mw"),
            F.round(F.max(F.abs("z_score")), 4).alias("max_abs_z_score"),
        )
        .orderBy("turbine_id")
    )

## 8. Storage — Delta MERGE (Upsert)

Write results to Delta tables using MERGE to handle incremental appends. New records are inserted; existing records (matched by natural key) are updated in place.

In [0]:
def merge_to_delta(df: DataFrame, table_name: str, key_columns: list) -> None:
    """
    Upsert a DataFrame into a Delta table. If the table does not exist it
    is created; otherwise records are matched on `key_columns` — matched
    rows are updated and new rows are inserted.
    """
    if not spark.catalog.tableExists(table_name):
        df.write.format("delta").saveAsTable(table_name)
        print(f"Created '{table_name}' with {df.count()} rows")
        return

    merge_condition = " AND ".join(
        [f"target.{col} = source.{col}" for col in key_columns]
    )

    delta_table = DeltaTable.forName(spark, table_name)

    (
        delta_table.alias("target")
        .merge(df.alias("source"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    total = spark.table(table_name).count()
    print(f"Merged into '{table_name}' — table now has {total} rows")

## 9. Run Pipeline (Incremental)

Load all CSVs, detect new records via anti-join, then clean, summarise, and merge only the incremental data into existing Delta tables.

In [0]:
raw_df = load_all(DATA_FILES)
print(f"Ingested {raw_df.count()} raw records from {raw_df.select('turbine_id').distinct().count()} turbines")

new_df = extract_new_records(raw_df, TABLE_CLEANED)
new_count = new_df.count()
print(f"New records to process: {new_count}")

if new_count == 0:
    print("No new data — pipeline has nothing to process.")
    dbutils.notebook.exit("NO_NEW_DATA")

display(new_df.limit(10))

Loaded 3720 records from data_group_1.csv
Loaded 3720 records from data_group_2.csv
Loaded 3720 records from data_group_3.csv
Combined dataset: 11160 records, 15 turbines
Ingested 11160 raw records from 15 turbines
Table 'hive_metastore.wind_turbine.cleaned_readings' does not exist yet — all records are new
New records to process: 11160


timestamp,turbine_id,wind_speed,wind_direction,power_output
2022-03-01T00:00:00Z,1,11.8,169.0,2.7
2022-03-01T01:00:00Z,1,11.6,152.0,4.4
2022-03-01T02:00:00Z,1,13.8,73.0,2.9
2022-03-01T03:00:00Z,1,10.5,61.0,1.8
2022-03-01T04:00:00Z,1,9.1,209.0,2.3
2022-03-01T05:00:00Z,1,12.4,116.0,2.2
2022-03-01T06:00:00Z,1,9.2,97.0,4.2
2022-03-01T07:00:00Z,1,10.1,87.0,4.0
2022-03-01T08:00:00Z,1,12.6,150.0,1.6
2022-03-01T09:00:00Z,1,9.1,31.0,2.7


In [0]:
cleaned_df = clean(new_df)
cleaned_df.cache()

print(f"Cleaned dataset: {cleaned_df.count()} records")
display(cleaned_df.describe())

Cleaned dataset: 11160 records


summary,turbine_id,wind_speed,wind_direction,power_output
count,11160,11160,11160,11160
mean,8.0,12.002724014336914,179.90394265232976,3.0019265232974903
stddev,4.32068738245902,1.7353660714427153,103.35351302055336,0.8698311327250574
min,1,9.0,0.0,1.5
max,15,15.0,359.0,4.5


In [0]:
daily_summary_df = compute_daily_summary(cleaned_df)
overall_summary_df = compute_overall_summary(cleaned_df)

print("Overall Statistics per Turbine:")
display(overall_summary_df)

Generated 465 daily summaries
Overall Statistics per Turbine:


turbine_id,min_power,max_power,mean_power,std_power,total_records
1,1.5,4.5,3.0164,0.8572,744
2,1.5,4.5,2.9836,0.8745,744
3,1.5,4.5,2.98,0.8626,744
4,1.5,4.5,2.9464,0.887,744
5,1.5,4.5,3.0165,0.8659,744
6,1.5,4.5,2.9839,0.8741,744
7,1.5,4.5,3.0126,0.8764,744
8,1.5,4.5,2.9848,0.8918,744
9,1.5,4.5,3.0028,0.8743,744
10,1.5,4.5,3.006,0.8663,744


In [0]:
anomalies_df = detect_anomalies(cleaned_df)
anomaly_summary_df = summarise_anomalies(anomalies_df)

print(f"Total anomalous readings: {anomalies_df.count()}")
print(f"Turbines with anomalies: {anomalies_df.select('turbine_id').distinct().count()}")
print()
print("Anomaly Summary per Turbine:")
display(anomaly_summary_df)

Detected 58 anomalous readings across 14 turbines
Total anomalous readings: 58
Turbines with anomalies: 14

Anomaly Summary per Turbine:


turbine_id,anomaly_count,avg_deviation_mw,max_abs_z_score
1,4,0.0583,2.3404
2,4,0.8313,2.2973
3,5,-0.9042,2.2876
4,8,1.2786,2.494
6,3,0.6264,2.1617
7,3,0.6875,2.0918
8,3,1.5403,2.3667
9,5,-0.9742,2.2379
10,6,0.5319,2.1542
11,5,1.6516,2.2094


In [0]:
print("Sample anomalous readings:")
display(anomalies_df.limit(10))

Sample anomalous readings:


timestamp,turbine_id,power_output,wind_speed,daily_mean,daily_std,z_score,deviation_mw
2022-03-05T21:00:00Z,1,1.6,14.0,3.2458,0.715,-2.3017,-1.6458
2022-03-11T09:00:00Z,1,4.4,11.8,2.575,0.7798,2.3404,1.825
2022-03-26T11:00:00Z,1,4.5,12.3,2.8167,0.8354,2.0149,1.6833
2022-03-29T09:00:00Z,1,1.5,9.0,3.1292,0.8111,-2.0087,-1.6292
2022-03-02T22:00:00Z,2,4.2,13.1,2.4667,0.7545,2.2973,1.7333
2022-03-22T10:00:00Z,2,4.5,13.7,2.9208,0.7835,2.0155,1.5792
2022-03-28T11:00:00Z,2,4.5,9.6,2.8583,0.814,2.0169,1.6417
2022-03-31T08:00:00Z,2,1.6,9.5,3.2292,0.7123,-2.2872,-1.6292
2022-03-03T02:00:00Z,3,1.8,9.2,3.2542,0.6692,-2.173,-1.4542
2022-03-03T05:00:00Z,3,1.9,13.9,3.2542,0.6692,-2.0236,-1.3542


## 10. Merge Results into Delta Tables

Upsert cleaned readings (keyed on turbine_id + timestamp), daily summaries (keyed on turbine_id + date), and anomalies (keyed on turbine_id + timestamp).

In [0]:
merge_to_delta(cleaned_df, TABLE_CLEANED, ["turbine_id", "timestamp"])
merge_to_delta(daily_summary_df, TABLE_DAILY_SUMMARY, ["turbine_id", "date"])
merge_to_delta(anomalies_df, TABLE_ANOMALIES, ["turbine_id", "timestamp"])

print("All pipeline outputs merged into Delta tables.")

Created 'hive_metastore.wind_turbine.cleaned_readings' with 11160 rows
Created 'hive_metastore.wind_turbine.daily_summary' with 465 rows
Created 'hive_metastore.wind_turbine.anomalies' with 58 rows
All pipeline outputs merged into Delta tables.


## 11. Verify Stored Data

In [0]:
for table in [TABLE_CLEANED, TABLE_DAILY_SUMMARY, TABLE_ANOMALIES]:
    count = spark.table(table).count()
    print(f"  {table}: {count} rows")

print("\nSample from daily_summary:")
display(spark.table(TABLE_DAILY_SUMMARY).limit(10))

  hive_metastore.wind_turbine.cleaned_readings: 11160 rows
  hive_metastore.wind_turbine.daily_summary: 465 rows
  hive_metastore.wind_turbine.anomalies: 58 rows

Sample from daily_summary:


turbine_id,date,min_power,max_power,mean_power,record_count
1,2022-03-01,1.6,4.4,2.975,24
1,2022-03-02,1.9,4.5,3.2375,24
1,2022-03-03,1.6,4.4,2.925,24
1,2022-03-04,1.5,4.4,2.9875,24
1,2022-03-05,1.6,4.3,3.2458,24
1,2022-03-06,1.5,4.4,2.9583,24
1,2022-03-07,1.9,4.3,3.2833,24
1,2022-03-08,1.7,4.5,3.2917,24
1,2022-03-09,1.5,4.5,2.8375,24
1,2022-03-10,1.7,4.3,3.1208,24


## 12. Unit Tests

Inline tests to validate each pipeline stage using Spark DataFrames.

In [0]:
def _make_spark_df(rows):
    """Helper to create a test Spark DataFrame from list of tuples."""
    return spark.createDataFrame(rows, schema=CSV_SCHEMA)


def run_tests():
    base_ts = datetime(2022, 3, 1, 0, 0, 0)
    sample_rows = []
    for tid in [1, 2]:
        for h in range(48):
            sample_rows.append((
                base_ts + timedelta(hours=h), tid,
                round(9 + 6 * np.random.random(), 1),
                float(np.random.randint(0, 360)),
                round(1.5 + 3 * np.random.random(), 1),
            ))
    sample = _make_spark_df(sample_rows)

    # -- Duplicate removal --
    dup_row = sample_rows[0]
    dup_df = _make_spark_df(sample_rows + [dup_row])
    deduped = remove_duplicates(dup_df)
    assert deduped.count() == sample.count(), "Duplicate removal failed"

    # -- Range validation --
    bad_rows = [(datetime(2022, 3, 1), 1, 10.0, 180.0, -5.0)]
    bad_df = _make_spark_df(bad_rows)
    ranged = replace_out_of_range(bad_df)
    null_count = ranged.filter(F.col("power_output").isNull()).count()
    assert null_count == 1, "Range check failed for negative power"

    bad_wind = [(datetime(2022, 3, 1), 1, 500.0, 180.0, 3.0)]
    ranged_w = replace_out_of_range(_make_spark_df(bad_wind))
    assert ranged_w.filter(F.col("wind_speed").isNull()).count() == 1, "Range check failed for extreme wind"

    # -- Imputation --
    gap_rows = [
        (datetime(2022, 3, 1, 0), 1, 10.0, 180.0, 2.0),
        (datetime(2022, 3, 1, 1), 1, None, 190.0, None),
        (datetime(2022, 3, 1, 2), 1, 12.0, 200.0, 4.0),
    ]
    gap_df = _make_spark_df(gap_rows)
    filled = impute_missing(gap_df)
    null_total = filled.filter(F.col("wind_speed").isNull() | F.col("power_output").isNull()).count()
    assert null_total == 0, "Imputation left nulls"

    # -- Full clean produces no nulls --
    dirty_rows = list(sample_rows)
    dirty_rows[5] = (dirty_rows[5][0], dirty_rows[5][1], dirty_rows[5][2], dirty_rows[5][3], None)
    dirty_rows[15] = (dirty_rows[15][0], dirty_rows[15][1], dirty_rows[15][2], dirty_rows[15][3], -999.0)
    dirty_df = _make_spark_df(dirty_rows)
    cleaned = clean(dirty_df)
    for col in ["wind_speed", "wind_direction", "power_output"]:
        assert cleaned.filter(F.col(col).isNull()).count() == 0, f"Full clean left nulls in {col}"

    # -- Daily summary: min <= mean <= max --
    daily = compute_daily_summary(sample)
    violations = daily.filter(
        (F.col("min_power") > F.col("mean_power")) | (F.col("mean_power") > F.col("max_power"))
    ).count()
    assert violations == 0, "min > mean or mean > max in summary"

    # -- Overall summary: one row per turbine --
    overall = compute_overall_summary(sample)
    assert overall.count() == sample.select("turbine_id").distinct().count(), "Wrong turbine count"

    # -- Anomaly detection: spike gets flagged --
    spike_rows = [
        (base_ts + timedelta(hours=h), 1, 10.0, 180.0, 3.0 if h < 23 else 10.0)
        for h in range(24)
    ]
    spike_anomalies = detect_anomalies(_make_spark_df(spike_rows), threshold=2.0)
    assert spike_anomalies.count() > 0, "Spike not detected"

    # -- Anomaly detection: uniform data has no anomalies --
    flat_rows = [
        (base_ts + timedelta(hours=h), 1, 10.0, 180.0, 3.0)
        for h in range(24)
    ]
    flat_anomalies = detect_anomalies(_make_spark_df(flat_rows), threshold=2.0)
    assert flat_anomalies.count() == 0, "False anomalies on uniform data"

    # -- Delta MERGE: initial write creates table, second merge is idempotent --
    test_table = f"{CATALOG}.{SCHEMA}._test_merge"
    spark.sql(f"DROP TABLE IF EXISTS {test_table}")

    batch_1 = _make_spark_df(sample_rows[:48])
    merge_to_delta(batch_1, test_table, ["turbine_id", "timestamp"])
    assert spark.table(test_table).count() == 48, "Initial merge failed"

    batch_2 = _make_spark_df(sample_rows[24:72])
    merge_to_delta(batch_2, test_table, ["turbine_id", "timestamp"])
    assert spark.table(test_table).count() == 72, "Incremental merge duplicated rows"

    # -- Incremental detection: anti-join filters already-stored records --
    full_raw = _make_spark_df(sample_rows[:72])
    new_only = extract_new_records(full_raw, test_table)
    assert new_only.count() == 0, "Anti-join should return 0 for fully loaded data"

    extra_rows = [(base_ts + timedelta(hours=99), 1, 10.0, 180.0, 3.0)]
    with_extra = _make_spark_df(sample_rows[:72] + extra_rows)
    new_only = extract_new_records(with_extra, test_table)
    assert new_only.count() == 1, "Anti-join should return 1 new record"

    spark.sql(f"DROP TABLE IF EXISTS {test_table}")

    print("All tests passed.")


np.random.seed(42)
run_tests()

Removed 1 duplicate records
Generated 4 daily summaries
Detected 1 anomalous readings across 1 turbines
Detected 0 anomalous readings across 0 turbines
Created 'hive_metastore.wind_turbine._test_merge' with 48 rows
Merged into 'hive_metastore.wind_turbine._test_merge' — table now has 72 rows
Identified 0 new records (out of 72 in CSVs)
Identified 1 new records (out of 73 in CSVs)
All tests passed.
